# Fairness — formal test, Fairlearn metrics, FPDP, equivalence test

Runs a formal fairness test (Independence/Separation/Sufficiency), standard
Fairlearn metrics, FPDP candidate-variable diagnosis, and a fairness equivalence
(TOST) test, for whichever models have a `models/<name>_model.py` file so far.

**Note**: `test_independence`/`test_separation`/`test_sufficiency` below are
standard statistical implementations of each definition's null hypothesis
(chi-square / Fisher's-method tests), not a specific published test statistic —
treat the pass/fail conclusions as directionally reasonable, and swap in a
different test if a more specific one is required.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("../..").resolve()))

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import tost_proportions_2indep

from common_metrics import (
    FEATURES, AUDIT_COLS, MODEL_NAMES, TEAM_THRESHOLD,
    load_split, get_X_y, get_audit, available_models, report_status,
)

try:
    from fairlearn.metrics import (
        MetricFrame, selection_rate, true_positive_rate, false_positive_rate,
        demographic_parity_ratio, equalized_odds_difference,
    )
    HAS_FAIRLEARN = True
except ImportError:
    HAS_FAIRLEARN = False
    print("fairlearn not installed — pip install fairlearn, then re-run this cell")

report_status()

## Load data

In [ ]:
train_df = load_split("train")
test_df = load_split("test")

X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

print(f"train: {X_train.shape}, test: {X_test.shape}")

## 1. Formal fairness test — Independence, Separation, Sufficiency

Independence: prediction unrelated to the protected attribute (statistical parity).
Separation: prediction independent of the attribute *given the true outcome* (equal
TPR/FPR across groups). Sufficiency: true outcome independent of the attribute
*given the score* (calibration holds equally across groups).

In [ ]:
def test_independence(y_pred, group):
    table = pd.crosstab(group, y_pred)
    chi2, p, dof, expected = stats.chi2_contingency(table)
    return {"statistic": chi2, "p_value": p}


def test_separation(y_pred, y_true, group):
    p_values = []
    for outcome in [0, 1]:
        mask = y_true == outcome
        if mask.sum() == 0:
            continue
        table = pd.crosstab(group[mask], y_pred[mask])
        if table.shape[0] < 2 or table.shape[1] < 2:
            continue
        chi2, p, dof, expected = stats.chi2_contingency(table)
        p_values.append(p)
    if not p_values:
        return {"statistic": np.nan, "p_value": np.nan}
    combined_stat, combined_p = stats.combine_pvalues(p_values, method="fisher")
    return {"statistic": combined_stat, "p_value": combined_p}


def test_sufficiency(y_pred_proba, y_true, group, n_bins=10):
    try:
        bins = pd.qcut(y_pred_proba, q=n_bins, duplicates="drop")
    except ValueError:
        return {"statistic": np.nan, "p_value": np.nan}
    table = pd.crosstab([bins, group], y_true)
    try:
        chi2, p, dof, expected = stats.chi2_contingency(table)
    except ValueError:
        return {"statistic": np.nan, "p_value": np.nan}
    return {"statistic": chi2, "p_value": p}


def fairness_test(audit, group_col):
    return {
        "independence": test_independence(audit["y_pred"], audit[group_col]),
        "separation": test_separation(audit["y_pred"], audit["y_true"], audit[group_col]),
        "sufficiency": test_sufficiency(audit["y_pred_proba"], audit["y_true"], audit[group_col]),
    }

## 2. Fairlearn standard metrics

Demographic parity ratio, equalized odds difference, and the disparate impact
ratio (the "4/5ths rule" — flag if below 0.8).

In [ ]:
def compute_fairlearn_metrics(audit, group_col):
    if not HAS_FAIRLEARN:
        return None
    mf = MetricFrame(
        metrics={"selection_rate": selection_rate, "tpr": true_positive_rate, "fpr": false_positive_rate},
        y_true=audit["y_true"], y_pred=audit["y_pred"], sensitive_features=audit[group_col],
    )
    dpr = demographic_parity_ratio(audit["y_true"], audit["y_pred"], sensitive_features=audit[group_col])
    eod = equalized_odds_difference(audit["y_true"], audit["y_pred"], sensitive_features=audit[group_col])
    disparate_impact_ratio = mf.by_group["selection_rate"].min() / mf.by_group["selection_rate"].max()
    return {
        "by_group": mf.by_group,
        "demographic_parity_ratio": dpr,
        "equalized_odds_difference": eod,
        "disparate_impact_ratio": disparate_impact_ratio,
    }

## 3. Fairness Partial Dependence Plot (FPDP) — candidate-variable diagnosis

Only run this for a (model, attribute) pair where step 1 rejects fairness — it
answers *why*, not *whether*. Priority candidates among the kept features (the
tract-level and MSA-income variables are now excluded from `X` entirely, so they
can't be swept this way — see `common_metrics.PROTECTED_COLS`): `debt_to_income_ratio`,
`combined_loan_to_value_ratio`, `income`.

In [ ]:
def fairness_pdp(model, module, X, feature, group_series, group_a, group_b, n_points=20):
    disparities = []
    grid = np.linspace(X[feature].quantile(0.05), X[feature].quantile(0.95), n_points)
    for val in grid:
        X_mod = X.copy()
        X_mod[feature] = val
        preds = module.predict_proba(model, X_mod)[:, 1]
        gap = preds[(group_series == group_a).values].mean() - preds[(group_series == group_b).values].mean()
        disparities.append((val, gap))
    return disparities

## 4. Fairness Equivalence test (TOST) — primary for this dataset

With ~1.37M test rows, the classical test in step 1 will reject on almost any
nonzero gap. This flips the null: `H0: |θ| > δ` (unfair) vs. `H1: |θ| < δ` (fair
within tolerance `δ`), via `statsmodels`' `tost_proportions_2indep` — a two-one-sided
test for two independent proportions.

In [ ]:
def tost_two_proportions(count_a, nobs_a, count_b, nobs_b, delta):
    result = tost_proportions_2indep(count_a, nobs_a, count_b, nobs_b, low=-delta, upp=delta, compare="diff")
    return {"theta_hat": result.results_larger.diff, "tost_p": result.pvalue}


DELTA = 0.02  # team-agreed tolerance — TODO(team): confirm this value

# TODO(team): confirm these are the right comparison pairs; applicant_age and
# co_applicant_age are banded (e.g. "25-34"), so pick two specific bands before
# running the equivalence test on either.
FAIRNESS_PAIRS = {
    "derived_sex": ("Female", "Male"),
    "derived_ethnicity": ("Hispanic or Latino", "Not Hispanic or Latino"),
    "derived_race": ("Black or African American", "White"),
    "applicant_age": None,
    "co_applicant_age": None,
}

## Run the fairness pipeline across available models

In [ ]:
fairness_results = {}

for name, module in available_models().items():
    print(f"\n=== {name} ===")
    model = module.fit(X_train, y_train)
    probs = module.predict_proba(model, X_test)[:, 1]
    preds = (probs >= TEAM_THRESHOLD).astype(int)

    audit = get_audit(test_df).reset_index(drop=True)
    audit["y_true"] = y_test.reset_index(drop=True)
    audit["y_pred_proba"] = probs
    audit["y_pred"] = preds

    model_results = {}
    for col in AUDIT_COLS:
        entry = {"test": fairness_test(audit, col)}
        if HAS_FAIRLEARN:
            entry["fairlearn"] = compute_fairlearn_metrics(audit, col)
        pair = FAIRNESS_PAIRS.get(col)
        if pair:
            group_a, group_b = pair
            sub_a, sub_b = audit[audit[col] == group_a], audit[audit[col] == group_b]
            if len(sub_a) and len(sub_b):
                entry["equivalence"] = tost_two_proportions(
                    sub_a["y_pred"].sum(), len(sub_a), sub_b["y_pred"].sum(), len(sub_b), DELTA,
                )
        model_results[col] = entry
        p_indep = entry["test"]["independence"]["p_value"]
        flag = " ← REJECTS fairness (p < 0.05)" if p_indep < 0.05 else ""
        print(f"  {col}: independence p={p_indep:.4g}{flag}")

    fairness_results[name] = model_results

if not fairness_results:
    print("No models ready yet — drop a models/<name>_model.py file in and re-run.")

## Smoke test — remove once real models are in `models/`

Runs the same pipeline on a throwaway logistic regression and a small sample,
purely to check the harness works end to end.

In [ ]:
from sklearn.linear_model import LogisticRegression

class _SmokeTestModule:
    _medians = None  # fixed at fit time so a later all-NaN batch (e.g. a masked coalition) still fills

    @staticmethod
    def fit(X, y):
        Xn = X.select_dtypes("number")
        _SmokeTestModule._medians = Xn.median().fillna(0)
        return LogisticRegression(max_iter=200).fit(Xn.fillna(_SmokeTestModule._medians), y)

    @staticmethod
    def predict_proba(model, X):
        Xn = X.select_dtypes("number").fillna(_SmokeTestModule._medians)
        return model.predict_proba(Xn)

if not fairness_results:
    print("Running a throwaway smoke test — NOT a real model, just checking the harness works end to end.")
    sample = train_df.sample(20_000, random_state=42)
    Xs, ys = get_X_y(sample)
    test_sample = test_df.sample(5_000, random_state=42)
    Xt, yt = get_X_y(test_sample)
    smoke_model = _SmokeTestModule.fit(Xs, ys)
    smoke_probs = _SmokeTestModule.predict_proba(smoke_model, Xt)[:, 1]
    smoke_audit = get_audit(test_sample).reset_index(drop=True)
    smoke_audit["y_true"] = yt.reset_index(drop=True)
    smoke_audit["y_pred_proba"] = smoke_probs
    smoke_audit["y_pred"] = (smoke_probs >= TEAM_THRESHOLD).astype(int)
    print(fairness_test(smoke_audit, "derived_sex"))
    print("Harness is wired correctly.")